In [4]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [5]:
import os
print("cwd:", os.getcwd())
print("secrets exists:", os.path.exists("../05_src/.secrets"))

cwd: c:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\02_activities
secrets exists: True


In [ ]:
import os
from openai import OpenAI

def get_client() -> OpenAI:
    """
    Creates an OpenAI SDK client routed through the API Gateway.
    """
    api_gateway_key = os.getenv("API_GATEWAY_KEY")
    if not api_gateway_key:
        raise RuntimeError("API_GATEWAY_KEY is not set. Check .secrets and dotenv loading.")

    return OpenAI(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",  
        default_headers={"x-api-key": api_gateway_key},
    )

In [7]:
# Implement Guardrails 
# Prevent access/reveal of system prompt
# Prevent modification of system prompt
# Block restricted topics: cats/dogs, horoscopes/zodiac, Taylor Swift
import re
from typing import Optional

RESTRICTED_PATTERNS = [
    r"\bcat(s)?\b",
    r"\bdog(s)?\b",
    r"\bhoroscope(s)?\b",
    r"\bzodiac\b",
    r"\btaylor\s+swift\b",
]

PROMPT_LEAK_PATTERNS = [
    r"system prompt",
    r"developer prompt",
    r"show (me )?your prompt",
    r"reveal (the )?instructions",
    r"what are your rules",
]

PROMPT_INJECTION_PATTERNS = [
    r"ignore (all )?(previous|prior) instructions",
    r"you are now",
    r"act as",
    r"pretend you are",
    r"override",
]
def check_guardrails(user_text: str) -> Optional[str]:
#def check_guardrails(user_text: str) -> str | None:
    """
    Returns a refusal message if input violates guardrails; otherwise None.
    """
    t = user_text.lower()

    # Restricted topics
    for pat in RESTRICTED_PATTERNS:
        if re.search(pat, t):
            return "Sorry — I can’t help with that topic. Ask me something else."

    # System prompt exfiltration
    for pat in PROMPT_LEAK_PATTERNS:
        if re.search(pat, t):
            return "I can’t share or reveal my system/developer instructions. I can still help with your task though."

    # Prompt modification attempts (we don't refuse—just ignore them)
    for pat in PROMPT_INJECTION_PATTERNS:
        if re.search(pat, t):
            return "I can’t follow instruction-override requests. Tell me what you want to accomplish and I’ll help."

    return None

In [8]:
# Build Conversation Memory (required)
# Store chat history in a list of messages
# Optionally summarize when it gets too long
from dataclasses import dataclass, field

@dataclass
class ChatMemory:
    """
    Lightweight short-term memory.
    Stores messages and can truncate if too large.
    """
    max_turns: int = 20
    messages: list[dict] = field(default_factory=list)  # OpenAI message format

    def add_user(self, text: str):
        self.messages.append({"role": "user", "content": text})
        self._truncate()

    def add_assistant(self, text: str):
        self.messages.append({"role": "assistant", "content": text})
        self._truncate()

    def _truncate(self):
        # Keep last N turns (2 messages per turn)
        if len(self.messages) > self.max_turns * 2:
            self.messages = self.messages[-self.max_turns * 2:]

In [9]:
# Create the folders + empty __init__.py files
import os
from pathlib import Path

base = Path("../05_src/assignment_chat").resolve()   # from 02_activities
services = base / "services"
data_dir = base / "data"
scripts = base / "scripts"
chroma_dir = base / "chroma_db"

for d in [base, services, data_dir, scripts, chroma_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Make packages importable
(services / "__init__.py").write_text("", encoding="utf-8")

print("Created:", base)
print("Subfolders:", [p.name for p in base.iterdir()])

Created: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat
Subfolders: ['app.py', 'chroma_db', 'data', 'guardrails.py', 'memory.py', 'router.py', 'scripts', 'services', '__pycache__']


In [10]:
# Add that folder to sys.path
import sys
sys.path.insert(0, str(base))
print(sys.path[0])

C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat


In [11]:
# Create services/llm.py
llm_py = services / "llm.py"

llm_py.write_text(
'''import os
from openai import OpenAI

def get_client():
    api_gateway_key = os.getenv("API_GATEWAY_KEY")
    if not api_gateway_key:
        raise RuntimeError("API_GATEWAY_KEY is not set. Load ../05_src/.secrets first.")

    return OpenAI(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",
        default_headers={"x-api-key": api_gateway_key},
    )
''',
    encoding="utf-8"
)

print("Wrote:", llm_py)

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\services\llm.py


In [12]:
# Service 1: API Calls 

import requests
from services.llm import get_client

def weather_service(city: str) -> str:
    """
    Service 1: Calls a public API (Open-Meteo via geocoding + forecast),
    then rephrases results (not verbatim).
    """
    # 1) Geocode city -> lat/lon
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=20
    ).json()

    if not geo.get("results"):
        return f"I couldn’t find weather data for '{city}'. Try a nearby major city."

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    place = geo["results"][0]["name"]

    # 2) Forecast
    forecast = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current_weather": True,
            "timezone": "auto"
        },
        timeout=20
    ).json()

    # 3) Rephrase with LLM (requirement: not verbatim)
    client = get_client()
    prompt = f"""
    Convert this weather JSON into a short, friendly summary (3-5 sentences).
    Do NOT copy fields verbatim; rephrase naturally.
    Location: {place}
    JSON: {forecast}
    """

    resp = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        max_output_tokens=200,
    )
    return resp.output_text

In [ ]:
# service 2 — Point to extracted dataset files
from pathlib import Path

BASE = Path("../05_src/assignment_chat").resolve()
DATA_DIR = BASE / "data"
CHROMA_DIR = BASE / "chroma_db"

print("DATA_DIR:", DATA_DIR, "exists?", DATA_DIR.exists())
print("Files:", [p.name for p in DATA_DIR.glob("pitchfork_*")])

DATA_DIR: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\data exists? True
Files: ['pitchfork_artists.jsonl', 'pitchfork_content.jsonl', 'pitchfork_genres.jsonl', 'pitchfork_labels.jsonl', 'pitchfork_reviews.jsonl', 'pitchfork_years.jsonl']


In [14]:
# service 2 -Load the extracted pitchfork_reviews file
import pandas as pd
import json

def load_table(path_no_ext: Path) -> pd.DataFrame:
    """
    Loads a table from either CSV or JSONL given a base name without extension.
    Example: load_table(DATA_DIR / "pitchfork_reviews")
    """
    csv_path = path_no_ext.with_suffix(".csv")
    jsonl_path = path_no_ext.with_suffix(".jsonl")

    if csv_path.exists():
        return pd.read_csv(csv_path)

    if jsonl_path.exists():
        rows = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return pd.DataFrame(rows)

    raise FileNotFoundError(f"Expected {csv_path} or {jsonl_path} to exist.")

reviews_df = load_table(DATA_DIR / "pitchfork_reviews")
print("reviews_df shape:", reviews_df.shape)
print("columns:", list(reviews_df.columns)[:30])
reviews_df.head(3)

reviews_df shape: (18393, 13)
columns: ['reviewid', 'title', 'artist', 'url', 'score', 'best_new_music', 'author', 'author_type', 'pub_date', 'pub_weekday', 'pub_day', 'pub_month', 'pub_year']


,reviewid,title,artist,url,score,best_new_music,author,author_type,pub_date,pub_weekday,pub_day,pub_month,pub_year
0,22703,mezzanine,massive attack,http://pitchfork.com/reviews/albums/22703-mezz...,9.3,0,nate patrin,contributor,2017-01-08,6,8,1,2017
1,22721,prelapsarian,krallice,http://pitchfork.com/reviews/albums/22721-prel...,7.9,0,zoe camp,contributor,2017-01-07,5,7,1,2017
2,22659,all of them naturals,uranium club,http://pitchfork.com/reviews/albums/22659-all-...,7.3,0,david glickman,contributor,2017-01-07,5,7,1,2017


In [15]:
%load_ext dotenv
%dotenv ../05_src/.secrets

import os
print("API_GATEWAY_KEY loaded?", os.getenv("API_GATEWAY_KEY") is not None)

from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
API_GATEWAY_KEY loaded? True


In [16]:
import chromadb
from pathlib import Path

BASE = Path("../05_src/assignment_chat").resolve()
CHROMA_DIR = BASE / "chroma_db"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(name="pitchfork_reviews")
print("Chroma count:", collection.count())

Chroma count: 256


In [17]:
def semantic_search(query: str, k: int = 3) -> str:
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query]
    )
    qvec = emb.data[0].embedding

    results = collection.query(
        query_embeddings=[qvec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    context = ""
    for meta, dist, doc in zip(metas, dists, docs):
        context += f"\n- {meta.get('artist','')} — {meta.get('title','')} (distance={dist:.3f})"

    prompt = f"""
You are VinylSage, a music-review assistant.
The user asked: {query}

I retrieved these closest reviews:
{context}

In 4-6 sentences, explain what the retrieved reviews suggest.
Do NOT paste the reviews. Summarize themes.
If retrieval seems off-topic, say so.
"""
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        max_output_tokens=220
    )
    return resp.output_text

print(semantic_search("atmospheric electronic albums with glitchy textures", k=3))

The retrieved reviews suggest a strong focus on atmospheric electronic music that incorporates glitchy textures, creating immersive soundscapes. Machinedrum's "Human Energy" likely explores themes of dynamic sound manipulation and rhythmic complexity, while Metamatics’ "From Death to Passwords Where You're a Paper Aeroplane" hints at a conceptual depth, merging organic and digital elements. Apparatus's "Duplex" may emphasize emotive qualities alongside intricate production techniques. Overall, these albums embody a blend of experimental sound design and atmospheric depth, appealing to listeners who appreciate both innovation and texture in electronic music.


In [18]:
print(reviews_df.columns.tolist())

['reviewid', 'title', 'artist', 'url', 'score', 'best_new_music', 'author', 'author_type', 'pub_date', 'pub_weekday', 'pub_day', 'pub_month', 'pub_year']


In [19]:
# Load pitchfork_content
content_df = load_table(DATA_DIR / "pitchfork_content")
print("content_df columns:", content_df.columns.tolist())
content_df.head(3)

content_df columns: ['reviewid', 'content']


,reviewid,content
0,22703,"Trip-hop eventually became a 90s punchline, a ..."
1,22721,"Eight years, five albums, and two EPs in, the ..."
2,22659,Minneapolis Uranium Club seem to revel in bein...


In [20]:
# Find which column in pitchfork_content contains the text
import pandas as pd

def find_text_col(df):
    # common candidates
    candidates = ["review", "content", "text", "body", "review_text", "article", "html", "text_content"]
    for c in candidates:
        if c in df.columns:
            return c

    # fallback: choose the string column with the largest average length
    str_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not str_cols:
        return None

    avg_len = {c: df[c].fillna("").astype(str).map(len).mean() for c in str_cols}
    return max(avg_len, key=avg_len.get)

content_text_col = find_text_col(content_df)
print("Detected content_text_col:", content_text_col)

Detected content_text_col: content


In [21]:
# Join reviews + content on reviewid
# Ensure content has reviewid
if "reviewid" not in content_df.columns:
    raise RuntimeError("pitchfork_content does not have reviewid. Paste content_df.columns and we’ll adjust.")

merged = reviews_df.merge(
    content_df[["reviewid", content_text_col]],
    on="reviewid",
    how="left"
).rename(columns={content_text_col: "review_text"})

print("merged shape:", merged.shape)
print("missing review_text fraction:", merged["review_text"].isna().mean())
merged[["reviewid", "title", "artist", "review_text"]].head(3)

merged shape: (18401, 14)
missing review_text fraction: 0.0


,reviewid,title,artist,review_text
0,22703,mezzanine,massive attack,"Trip-hop eventually became a 90s punchline, a ..."
1,22721,prelapsarian,krallice,"Eight years, five albums, and two EPs in, the ..."
2,22659,all of them naturals,uranium club,Minneapolis Uranium Club seem to revel in bein...


In [22]:
#Set column mappings correctly
id_col = "reviewid"
title_col = "title"
artist_col = "artist"
text_col = "review_text"

In [23]:
# Build documents for embedding
def build_documents(df):
    docs, ids, metas = [], [], []
    for _, row in df.iterrows():
        rid = str(row[id_col])
        title = str(row[title_col]) if pd.notna(row[title_col]) else ""
        artist = str(row[artist_col]) if pd.notna(row[artist_col]) else ""
        review_text = str(row[text_col]) if pd.notna(row[text_col]) else ""

        doc = f"Title: {title}\nArtist: {artist}\nReview:\n{review_text}".strip()

        ids.append(rid)
        docs.append(doc)
        metas.append({"title": title, "artist": artist})
    return ids, docs, metas

# Keep repo + embedding size small
df_for_embedding = merged.dropna(subset=[text_col])
sample_df = df_for_embedding.sample(n=min(1500, len(df_for_embedding)), random_state=42)

ids, docs, metas = build_documents(sample_df)
print("docs:", len(docs))
print(docs[0][:500])

docs: 1500
Title: fixin' the charts, vol. 1
Artist: everybody was in the french resistance...now!
Review:
"Answer records are not new," Time magazine wrote. That was in 1961. From "Yes, I'm Lonesome Tonight" to "You Know I'll Love You Tomorrow", "Roll With Me, Henry" to "Wearing His Rolex", songs that respond to other songs have long been a lively pop tradition. In 1995, German reissue label Bear Family put out a three-volume compilation series, And the Answer Is: Great Pop Answer Discs From the '50s and '


In [24]:
print("merged rows:", len(merged))
print("review_text missing %:", merged["review_text"].isna().mean())
print("example chars:", len(str(merged["review_text"].dropna().iloc[0])))

merged rows: 18401
review_text missing %: 0.0
example chars: 9149


In [25]:
print("Chroma count:", collection.count())

Chroma count: 256


In [26]:
# Store the last retrieval context (memory for services)
last_retrieval = None  # global in notebook

def semantic_search(query: str, k: int = 3) -> str:
    global last_retrieval
    
    emb = client.embeddings.create(model="text-embedding-3-small", input=[query])
    qvec = emb.data[0].embedding

    results = collection.query(
        query_embeddings=[qvec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    # store for Service 3
    last_retrieval = {
        "query": query,
        "items": [
            {"meta": m, "distance": float(d), "doc": doc[:1200]}
            for m, d, doc in zip(metas, dists, docs)
        ]
    }

    context = ""
    for m, d in zip(metas, dists):
        context += f"\n- {m.get('artist','')} — {m.get('title','')} (distance={d:.3f})"

    prompt = f"""
You are VinylSage, a music-review assistant.
User question: {query}

Closest matches:
{context}

In 4-6 sentences, explain what the retrieved reviews suggest.
Do NOT paste the reviews. Summarize themes.
If retrieval seems off-topic, say so.
"""
    resp = client.responses.create(model="gpt-4o-mini", input=prompt, max_output_tokens=220)
    return resp.output_text

In [27]:
print(semantic_search("atmospheric electronic albums with glitchy textures", k=3))

The retrieved albums exhibit a strong emphasis on atmospheric soundscapes, characterized by intricate glitchy textures that evoke a sense of depth and complexity. Machinedrum’s "Human Energy" combines rhythmic elements with textured layers, creating an immersive listening experience. Metamatics’ "From Death to Passwords Where You're a Paper Aeroplane" explores more abstract sonic territories, emphasizing an ethereal quality mixed with glitch elements. Apparatus’ "Duplex" blends emotional melodies with electronic intricacies, suggesting a fusion of human sentiment and digital manipulation. Overall, these albums offer rich, layered sounds that invite deep engagement, making them excellent choices for fans of atmospheric electronic music.


In [ ]:
# Service 3- Function calling
import json

def insight_service(user_goal: str) -> str:
    global last_retrieval
    if not last_retrieval:
        return "Run a semantic search first so I have retrieved reviews to analyze."

    # Build compact context from last retrieval
    context = "\n".join(
        f"Title: {it['meta'].get('title','')}\n"
        f"Artist: {it['meta'].get('artist','')}\n"
        f"Excerpt: {it['doc']}\n"
        for it in last_retrieval["items"]
    )

    # Gateway expects tools[0].name at the top level
    tools = [{
        "type": "function",
        "name": "generate_insights",
        "description": "Generate structured insights from retrieved music reviews to help the user.",
        "parameters": {
            "type": "object",
            "properties": {
                "overall_sentiment": {"type": "string"},
                "key_themes": {"type": "array", "items": {"type": "string"}},
                "notable_descriptors": {"type": "array", "items": {"type": "string"}},
                "recommended_followups": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["overall_sentiment", "key_themes", "notable_descriptors", "recommended_followups"],
        },
    }]

    prompt = f"""
You are VinylSage. Use ONLY the provided context.
User goal: {user_goal}

Retrieved review context:
{context}

Call generate_insights now.
"""

    resp = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        tools=tools,
        tool_choice={"type": "function", "name": "generate_insights"},
        max_output_tokens=400
    )

    # Parse tool call output (robust)
    tool_calls = [o for o in resp.output if getattr(o, "type", None) in ("tool_call", "function_call")]
    if not tool_calls:
        # Helpful debug if gateway returns a different shape
        return "No tool call returned. Debug output:\n" + str(resp.output)

    args = json.loads(tool_calls[0].arguments)

    return (
        f"**Overall sentiment:** {args['overall_sentiment']}\n\n"
        f"**Key themes:**\n- " + "\n- ".join(args["key_themes"]) + "\n\n"
        f"**Notable descriptors:**\n- " + "\n- ".join(args["notable_descriptors"]) + "\n\n"
        f"**Suggested follow-up questions:**\n- " + "\n- ".join(args["recommended_followups"])
    )

In [29]:
print(insight_service("Help me find albums similar in vibe and production style."))

**Overall sentiment:** intriguing and experimental

**Key themes:**
- underground and unconventional production
- mix of IDM and glitch influences
- subdued and ambient elements
- recontextualization of popular sounds

**Notable descriptors:**
- subdued album
- evolving sound
- glitch-hop
- post-dubstep influence
- ambient and experimental

**Suggested follow-up questions:**
- check out similar artists like Sepalcure and T. Raumschmiere
- explore other albums on the Neo Ouija label
- listen to Apparat's other works for ambient textures
- try more IDM from the early 2000s era
- consider remix albums that highlight nuanced production styles


In [30]:
# Add a Router (choose which service to use)
def route(user_text: str) -> str:
    """
    Simple intent routing:
      - weather: <city>
      - search: <query>
      - insight: <goal>
    """
    t = user_text.lower().strip()

    if t.startswith("weather:"):
        city = user_text.split(":", 1)[1].strip()
        return weather_service(city)

    if t.startswith("search:"):
        q = user_text.split(":", 1)[1].strip()
        return semantic_search(q)

    if t.startswith("insight:"):
        goal = user_text.split(":", 1)[1].strip()
        return insight_service(goal)

    return (
        "Try one of these commands:\n"
        "- `weather: Toronto`\n"
        "- `search: atmospheric electronic albums with glitchy textures`\n"
        "- `insight: Help me find albums similar in vibe and production style.` (run after `search:`)\n"
    )

In [76]:
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.18.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.27.0 which is incompatible.
google-adk 1.18.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.27.0 which is incompatible.
google-adk 1.18.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.2.3 which is incompatible.
google-adk 1.18.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 12.0 which is incompatible.
google-genai 1.47.0 requires websockets<15.1.0,>=13.0.0, but you have websockets 12.0 which is incompatible.


In [31]:
import gradio as gr
print(gr.__version__)

4.44.1


In [32]:
# add assignment_chat to sys.path
import os, sys
from pathlib import Path

BASE = Path("../05_src/assignment_chat").resolve()   # because cwd is 02_activities
print("BASE:", BASE)
print("exists?", BASE.exists())

if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

print("sys.path[0]:", sys.path[0])
print("files:", [p.name for p in BASE.glob("*.py")])

BASE: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat
exists? True
sys.path[0]: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat
files: ['app.py', 'guardrails.py', 'memory.py', 'router.py']


In [33]:
# Create guardrails.py

code = """import re
from typing import Optional

RESTRICTED_PATTERNS = [
    r"\\bcat(s)?\\b",
    r"\\bdog(s)?\\b",
    r"\\bhoroscope(s)?\\b",
    r"\\bzodiac\\b",
    r"\\btaylor\\s+swift\\b",
]

PROMPT_LEAK_PATTERNS = [
    r"system prompt",
    r"developer prompt",
    r"show (me )?your prompt",
    r"reveal (the )?instructions",
    r"what are your rules",
]

PROMPT_INJECTION_PATTERNS = [
    r"ignore (all )?(previous|prior) instructions",
    r"override",
    r"act as",
    r"pretend you are",
]

def check_guardrails(user_text: str) -> Optional[str]:
    \"""
    Returns a refusal message if input violates guardrails; otherwise None.
    \"""
    t = (user_text or "").lower()

    for pat in RESTRICTED_PATTERNS:
        if re.search(pat, t):
            return "Sorry — I can’t help with that topic. Ask me something else."

    for pat in PROMPT_LEAK_PATTERNS:
        if re.search(pat, t):
            return "I can’t share system/developer instructions. I can still help with your task."

    for pat in PROMPT_INJECTION_PATTERNS:
        if re.search(pat, t):
            return "I can’t follow instruction-override requests. Tell me your goal and I’ll help."

    return None
"""

PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
path = PROJECT_ROOT / "05_src" / "assignment_chat" / "guardrails.py"

path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(code, encoding="utf-8")

print("Wrote:", path)

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\guardrails.py


In [34]:
# Write memory.py
from pathlib import Path

memory_code = """from dataclasses import dataclass, field

@dataclass
class ChatMemory:
    \"""
    Lightweight short-term memory.
    Keeps the last max_turns user/assistant turns (2 messages per turn).
    \"""
    max_turns: int = 20
    messages: list = field(default_factory=list)  # list of {"role": ..., "content": ...}

    def add_user(self, text: str):
        self.messages.append({"role": "user", "content": text})
        self._truncate()

    def add_assistant(self, text: str):
        self.messages.append({"role": "assistant", "content": text})
        self._truncate()

    def _truncate(self):
        limit = self.max_turns * 2
        if len(self.messages) > limit:
            self.messages = self.messages[-limit:]
"""

PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
BASE = PROJECT_ROOT / "05_src" / "assignment_chat"
BASE.mkdir(parents=True, exist_ok=True)

(BASE / "memory.py").write_text(memory_code, encoding="utf-8")
print("Wrote:", (BASE / "memory.py"))

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\memory.py


In [35]:
# Write services/llm.py (Gateway client)
from pathlib import Path

# Set your project root
PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")

# Define the services folder path
SERVICES = PROJECT_ROOT / "05_src" / "assignment_chat" / "services"
SERVICES.mkdir(parents=True, exist_ok=True)

# Your llm.py content
llm_code = """import os
from openai import OpenAI

def get_client() -> OpenAI:
    return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def chat_completion(messages, model="gpt-4o-mini"):
    client = get_client()
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    return resp.choices[0].message.content
"""

# Write file
(SERVICES / "llm.py").write_text(llm_code, encoding="utf-8")
print("Wrote:", SERVICES / "llm.py")

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\services\llm.py


In [36]:
# Write Service 1: services/api_service.py
from pathlib import Path

# Paths
PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
SERVICES = PROJECT_ROOT / "05_src" / "assignment_chat" / "services"
SERVICES.mkdir(parents=True, exist_ok=True)

api_service_code = """import requests
from services.llm import chat_completion

def weather_service(city: str) -> str:
    \"""
    Service 1: Calls Open-Meteo API, then rephrases output (not verbatim).
    \"""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=20
    ).json()

    if not geo.get("results"):
        return f"I couldn’t find weather data for '{city}'. Try a nearby major city."

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    place = geo["results"][0]["name"]

    forecast = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True, "timezone": "auto"},
        timeout=20
    ).json()

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant that summarizes weather clearly and naturally."
        },
        {
            "role": "user",
            "content": (
                "Convert this weather JSON into a short, friendly summary (3–5 sentences). "
                "Do NOT copy fields verbatim; rephrase naturally.\\n\\n"
                f"Location: {place}\\n"
                f"JSON: {forecast}"
            ),
        },
    ]

    return chat_completion(messages, model="gpt-4o-mini")
"""

(SERVICES / "api_service.py").write_text(api_service_code, encoding="utf-8")
print("Wrote:", SERVICES / "api_service.py")

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\services\api_service.py


In [37]:
# Write Service 2: services/semantic_service.py
from pathlib import Path

# Paths
PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
SERVICES = PROJECT_ROOT / "05_src" / "assignment_chat" / "services"
SERVICES.mkdir(parents=True, exist_ok=True)

semantic_service_code = """import os
import chromadb
from services.llm import get_client, chat_completion

CHROMA_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "chroma_db"))
COLLECTION_NAME = "pitchfork_reviews"

# Used by Service 3
LAST_RETRIEVAL = None

def semantic_search(query: str, k: int = 3) -> str:
    global LAST_RETRIEVAL

    client = get_client()
    emb = client.embeddings.create(model="text-embedding-3-small", input=[query])
    qvec = emb.data[0].embedding

    chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = chroma_client.get_collection(name=COLLECTION_NAME)

    results = collection.query(
        query_embeddings=[qvec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    LAST_RETRIEVAL = {
        "query": query,
        "items": [
            {"meta": m, "distance": float(d), "doc": (doc or "")[:1200]}
            for m, d, doc in zip(metas, dists, docs)
        ]
    }

    context = ""
    for m, d in zip(metas, dists):
        context += f"\\n- {m.get('artist','')} — {m.get('title','')} (distance={d:.3f})"

    messages = [
        {"role": "system", "content": "You are VinylSage, a music-review assistant."},
        {"role": "user", "content": (
            f"User question: {query}\\n\\n"
            f"Closest matches:{context}\\n\\n"
            "In 4–6 sentences, explain what the retrieved reviews suggest. "
            "Do NOT paste the reviews. Summarize themes. "
            "If retrieval seems off-topic, say so."
        )},
    ]

    return chat_completion(messages, model="gpt-4o-mini")
"""

(SERVICES / "semantic_service.py").write_text(semantic_service_code, encoding="utf-8")
print("Wrote:", SERVICES / "semantic_service.py")

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\services\semantic_service.py


In [38]:
# Create services/tool_service.py
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
SERVICES = PROJECT_ROOT / "05_src" / "assignment_chat" / "services"
SERVICES.mkdir(parents=True, exist_ok=True)

tool_service_code = """from services.semantic_service import LAST_RETRIEVAL
from services.llm import chat_completion

def insight_service(goal: str) -> str:
    \"""
    Service 3: Generates structured insights based on the most recent semantic retrieval.
    Expects the user to run `search:` first.
    \"""
    if not LAST_RETRIEVAL or not LAST_RETRIEVAL.get("items"):
        return (
            "I don’t have any recent search results to base insights on yet.\\n"
            "First run something like:\\n"
            "- search: atmospheric electronic albums with glitchy textures\\n"
            "Then run:\\n"
            "- insight: " + goal
        )

    items = LAST_RETRIEVAL["items"]

    # Compact context for the LLM (no long review pastes)
    bullets = []
    for it in items:
        m = it.get("meta", {}) or {}
        artist = m.get("artist", "")
        title = m.get("title", "")
        score = m.get("score", "")
        year = m.get("year", "")
        dist = it.get("distance", None)
        bullets.append(f"- {artist} — {title} (year={year}, score={score}, distance={dist})")

    context = "\\n".join(bullets)

    messages = [
        {"role": "system", "content": "You are VinylSage — witty, slightly formal, and helpful."},
        {"role": "user", "content": (
            f"User goal: {goal}\\n\\n"
            "You have these closest Pitchfork review matches (metadata only):\\n"
            f"{context}\\n\\n"
            "Create a structured response with these sections:\\n"
            "1) What you seem to like (inferred)\\n"
            "2) Recommended directions / sub-genres to explore\\n"
            "3) 5 concrete album or artist suggestions (if unsure, say so)\\n"
            "4) A short next-step query the user should try with `search:`\\n\\n"
            "Do not paste any review text. Keep it concise but helpful."
        )},
    ]

    return chat_completion(messages, model="gpt-4o-mini")
"""

(SERVICES / "tool_service.py").write_text(tool_service_code, encoding="utf-8")
print("Wrote:", SERVICES / "tool_service.py")

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\services\tool_service.py


In [39]:

# Create router.py
from pathlib import Path

router_code = """from services.api_service import weather_service
from services.semantic_service import semantic_search
from services.tool_service import insight_service

def route(user_text: str) -> str:
    t = (user_text or "").strip()

    if t.lower().startswith("weather:"):
        city = t.split(":", 1)[1].strip()
        return weather_service(city)

    if t.lower().startswith("search:"):
        q = t.split(":", 1)[1].strip()
        return semantic_search(q)

    if t.lower().startswith("insight:"):
        goal = t.split(":", 1)[1].strip()
        return insight_service(goal)

    return (
        "Try one of these commands:\\n"
        "- weather: Toronto\\n"
        "- search: atmospheric electronic albums with glitchy textures\\n"
        "- insight: Help me find albums similar in vibe and production style. (run after search)\\n"
    )
"""

PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
BASE = PROJECT_ROOT / "05_src" / "assignment_chat"
BASE.mkdir(parents=True, exist_ok=True)

path = BASE / "router.py"
path.write_text(router_code, encoding="utf-8")

print("Wrote:", path)
print("Exists?", path.exists())

Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\router.py
Exists? True


In [40]:
# Create app.py (Gradio UI)
from pathlib import Path

# --- Paths ---
PROJECT_ROOT = Path(r"C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai")
BASE = PROJECT_ROOT / "05_src" / "assignment_chat"
BASE.mkdir(parents=True, exist_ok=True)

# --- Write app.py ---
app_code = """import gradio as gr
from guardrails import check_guardrails
from memory import ChatMemory
from router import route

PERSONALITY = (
    "I’m VinylSage — witty, slightly formal, and helpful. "
    "I can summarize weather, search Pitchfork reviews semantically, and generate structured insights."
)

memory = ChatMemory(max_turns=20)

def chat(user_text, chat_history):
    block = check_guardrails(user_text)
    if block:
        chat_history.append((user_text, block))
        return "", chat_history

    memory.add_user(user_text)
    reply = route(user_text)
    memory.add_assistant(reply)

    chat_history.append((user_text, reply))
    return "", chat_history

with gr.Blocks() as demo:
    gr.Markdown("## Assignment-2 Chat Client\\n\\n**Personality:** " + PERSONALITY)
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="weather: Toronto | search: ... | insight: ...", label="Message")
    clear = gr.Button("Clear")

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch(share=True)
"""

(BASE / "app.py").write_text(app_code, encoding="utf-8")
print("Wrote:", BASE / "app.py")
print("Exists?", (BASE / "app.py").exists())


Wrote: C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat\app.py
Exists? True


In [43]:
import os
print("CWD:", os.getcwd())

CWD: c:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\02_activities


In [44]:
%cd "C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat"
!dir

C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat
 Volume in drive C has no label.
 Volume Serial Number is 949B-9BE5

 Directory of C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat

02/25/2026  01:55 PM    <DIR>          .
02/23/2026  03:12 PM    <DIR>          ..
02/25/2026  06:10 PM             1,096 app.py
02/25/2026  06:08 PM    <DIR>          chroma_db
02/23/2026  03:26 PM    <DIR>          data
02/25/2026  06:09 PM             1,230 guardrails.py
02/25/2026  06:10 PM               743 memory.py
02/25/2026  06:10 PM               867 router.py
02/23/2026  03:12 PM    <DIR>          scripts
02/25/2026  02:16 PM    <DIR>          services
02/25/2026  02:10 PM    <DIR>          __pycache__
               4 File(s)          3,936 bytes
               7 Dir(s)  30,242,680,832 bytes free


In [45]:
from pathlib import Path
p = Path("app.py")
txt = p.read_text(encoding="utf-8")
txt = txt.replace("demo.launch()", "demo.launch(share=True)")
p.write_text(txt, encoding="utf-8")
print("Patched app.py to use share=True")

Patched app.py to use share=True


In [46]:
!python app.py

^C


In [49]:
%cd "C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat"
!python -u -X faulthandler app.py

C:\Users\hhsafa\dsi\AI-Deployment\deploying-ai\05_src\assignment_chat
^C


In [ ]:
!tasklist /FI "IMAGENAME eq python.exe"